# Model 1: TF-IDF + Naive Bayes -- evaluate

Loads the artifacts persisted by `train.ipynb` (no refitting here) and scores
them on the held-out test set. Reconstructs the same test set by rerunning
the identical `train_test_split` call -- deterministic given the same data,
`test_size`, `random_state`, and `stratify`, so no need to persist the split
itself.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('../sentences_top100.csv').dropna(subset=['content'])
data = data[data['content'].str.strip() != ''].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    data['content'],
    data['lecturer_id'],
    test_size=0.2,
    random_state=42,
    stratify=data['lecturer_id'],
)
len(X_test)

863553

In [2]:
import joblib

vectorizer = joblib.load('vectorizer.joblib')
clf = joblib.load('model.joblib')

X_test_tfidf = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_tfidf)

In [3]:
from sklearn.metrics import accuracy_score, classification_report

pd.set_option('display.max_rows', None)  # show all rows

baseline_acc = y_test.value_counts(normalize=True).iloc[0] # majority class
print('majority-class baseline accuracy:', round(baseline_acc, 4))
print('model accuracy:', round(accuracy_score(y_test, y_pred), 4))
print()

report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_dict.pop('accuracy')  # already printed above as 'model accuracy'
report = pd.DataFrame(report_dict).T
per_class = report.drop(['macro avg', 'weighted avg']).sort_values('support', ascending=False)
report_sorted = pd.concat([per_class, report.loc[['macro avg', 'weighted avg']]]).rename_axis('lecturer_id')
report_sorted.to_csv('evaluate.csv')
report_sorted

majority-class baseline accuracy: 0.1763
model accuracy: 0.4465



,precision,recall,f1-score,support
lecturer_id,,,,
1,0.551097,0.762049,0.639629,152216.0
3560,0.521750,0.536445,0.528995,119688.0
95,0.359111,0.802590,0.496202,83785.0
241,0.555563,0.369671,0.443943,68147.0
77,0.522278,0.406583,0.457225,35058.0
5573,0.242755,0.905149,0.382836,34064.0
477,0.520982,0.187155,0.275383,28789.0
8632,0.629702,0.186397,0.287647,25773.0
222,0.798849,0.220094,0.345107,24594.0
